# Data Quality: FHV - 8 Dimensiones - NYC TLC

Evaluacion de las 8 dimensiones de calidad de datos para el conjunto de datos FHV (For-Hire Vehicle) del NYC Taxi and Limousine Commission.

| Dimension | Descripcion |
|---|---|
| Completitud | Presencia de valores en campos obligatorios |
| Exactitud | Valores concordantes con catalogos conocidos |
| Consistencia | Coherencia logica entre campos relacionados |
| Integridad | Integridad referencial con el catalogo de zonas TLC |
| Razonabilidad | Valores dentro de rangos esperados |
| Oportunidad | Datos dentro del rango temporal valido |
| Unicidad | Ausencia de registros duplicados |
| Validez | Formatos correctos en campos clave |

In [ ]:
import sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sys.path.insert(0, '/home/jovyan/work')

from pyspark.sql import functions as F

# Tipo de vehiculo
VEHICLE = 'fhv'
RAW_PATH = f'/home/jovyan/work/data/raw/{VEHICLE}/'

# Diccionario para almacenar resultados de calidad
resultados_dq = {}

print(f'Tipo de vehiculo: {VEHICLE}')
print(f'Ruta de datos: {RAW_PATH}')

In [ ]:
from src.spark_utils import get_spark

spark = get_spark(app_name=f'DQ_{VEHICLE.upper()}')

# Lectura del conjunto de datos
df = spark.read.parquet(RAW_PATH).cache()

total_registros = df.count()
print(f'Total de registros cargados: {total_registros:,}')
print(f'Columnas disponibles: {df.columns}')
df.printSchema()


## Dimension 1: Completitud

**Pregunta:** Tenemos todos los datos necesarios?

Se verifica la presencia de valores no nulos en los campos obligatorios del esquema FHV.
El score es el promedio del porcentaje de completitud por campo obligatorio.

In [ ]:
# Campos obligatorios para FHV
campos_obligatorios = [
    'dispatching_base_num',
    'pickup_datetime',
    'dropOff_datetime',
    'PUlocationID',
    'DOlocationID'
]

print('--- Dimension 1: Completitud ---')
print(f'Total de registros: {total_registros:,}')
print()

porcentajes_completitud = []
registros_fallidos_completitud = 0

for campo in campos_obligatorios:
    nulos = df.filter(F.col(campo).isNull()).count()
    pct_completo = ((total_registros - nulos) / total_registros) * 100
    porcentajes_completitud.append(pct_completo)
    registros_fallidos_completitud += nulos
    print(f'  Campo "{campo}": {nulos:,} nulos | {pct_completo:.2f}% completo')

# Score: promedio de completitud por campo
score_completitud = sum(porcentajes_completitud) / len(porcentajes_completitud)

resultados_dq['Completitud'] = {
    'score': round(score_completitud, 2),
    'descripcion': 'Porcentaje promedio de completitud en campos obligatorios',
    'registros_fallidos': registros_fallidos_completitud
}

print()
print(f'Score Completitud: {score_completitud:.2f}%')
print(f'Registros fallidos (suma de nulos por campo): {registros_fallidos_completitud:,}')

## Dimension 2: Exactitud

**Pregunta:** Los valores son correctos?

Se verifica que los valores coincidan con los catalogos conocidos del TLC:
- `dispatching_base_num`: debe comenzar con 'B' (todos los numeros de base TLC validos comienzan con B)
- `SR_Flag`: debe ser nulo o '1' (indicador de viaje compartido)

In [ ]:
print('--- Dimension 2: Exactitud ---')
print()

# Regla 1: dispatching_base_num debe comenzar con 'B'

# --- OPTIMIZACION ONE-PASS (Auto-generada) ---
_exprs = [
    F.sum(F.when(F.col('dispatching_base_num').isNotNull() & (~F.col('dispatching_base_num').startswith('B')), 1).otherwise(0)).alias('falla_base_num'),
    F.sum(F.when(F.col('SR_Flag').isNotNull() & (~F.col('SR_Flag').isin(['1'])), 1).otherwise(0)).alias('falla_sr_flag')
]
_res = df.agg(*_exprs).collect()[0]
falla_base_num = _res['falla_base_num'] or 0
falla_sr_flag = _res['falla_sr_flag'] or 0
# ---------------------------------------------
# falla_base_num (Optimizado en One-Pass)
print(f'  dispatching_base_num no comienza con "B": {falla_base_num:,} registros')

# Regla 2: SR_Flag debe ser nulo o '1'
# falla_sr_flag (Optimizado en One-Pass)
print(f'  SR_Flag con valor invalido (no nulo ni "1"): {falla_sr_flag:,} registros')

# Registros que fallan CUALQUIERA de las reglas de exactitud
df_falla_exactitud = df.filter(
    (
        F.col('dispatching_base_num').isNotNull() &
        (~F.col('dispatching_base_num').startswith('B'))
    ) |
    (
        F.col('SR_Flag').isNotNull() &
        (~F.col('SR_Flag').isin(['1']))
    )
)
registros_fallidos_exactitud = df_falla_exactitud.count()
score_exactitud = ((total_registros - registros_fallidos_exactitud) / total_registros) * 100

resultados_dq['Exactitud'] = {
    'score': round(score_exactitud, 2),
    'descripcion': 'Porcentaje de registros cuyos valores coinciden con catalogos TLC',
    'registros_fallidos': registros_fallidos_exactitud
}

print()
print(f'Score Exactitud: {score_exactitud:.2f}%')
print(f'Registros fallidos: {registros_fallidos_exactitud:,}')

## Dimension 3: Consistencia

**Pregunta:** Los datos son coherentes entre si?

Se verifica la coherencia logica entre campos relacionados:
- `pickup_datetime` debe ser anterior a `dropOff_datetime`
- La duracion del viaje debe ser mayor a 0 segundos (no se permiten viajes de duracion cero)

In [ ]:
print('--- Dimension 3: Consistencia ---')
print()

# Regla 1: pickup_datetime < dropOff_datetime

# --- OPTIMIZACION ONE-PASS (Auto-generada) ---
_exprs = [
    F.sum(F.when(F.col('pickup_datetime').isNotNull() & F.col('dropOff_datetime').isNotNull() & (F.col('pickup_datetime') >= F.col('dropOff_datetime')), 1).otherwise(0)).alias('falla_orden_fechas'),
    F.sum(F.when(F.col('pickup_datetime').isNotNull() & F.col('dropOff_datetime').isNotNull() & (F.unix_timestamp('dropOff_datetime') - F.unix_timestamp('pickup_datetime') == 0), 1).otherwise(0)).alias('falla_duracion_cero')
]
_res = df.agg(*_exprs).collect()[0]
falla_orden_fechas = _res['falla_orden_fechas'] or 0
falla_duracion_cero = _res['falla_duracion_cero'] or 0
# ---------------------------------------------
# falla_orden_fechas (Optimizado en One-Pass)
print(f'  pickup_datetime >= dropOff_datetime (orden incorrecto): {falla_orden_fechas:,} registros')

# Regla 2: duracion del viaje > 0 segundos
# falla_duracion_cero (Optimizado en One-Pass)
print(f'  Viajes con duracion exactamente 0 segundos: {falla_duracion_cero:,} registros')

# Registros que fallan CUALQUIERA de las reglas de consistencia
df_falla_consistencia = df.filter(
    (
        F.col('pickup_datetime').isNotNull() &
        F.col('dropOff_datetime').isNotNull() &
        (F.col('pickup_datetime') >= F.col('dropOff_datetime'))
    ) |
    (
        F.col('pickup_datetime').isNotNull() &
        F.col('dropOff_datetime').isNotNull() &
        (F.unix_timestamp('dropOff_datetime') - F.unix_timestamp('pickup_datetime') == 0)
    )
)
registros_fallidos_consistencia = df_falla_consistencia.count()
score_consistencia = ((total_registros - registros_fallidos_consistencia) / total_registros) * 100

resultados_dq['Consistencia'] = {
    'score': round(score_consistencia, 2),
    'descripcion': 'Porcentaje de registros con coherencia logica entre campos de fecha y duracion',
    'registros_fallidos': registros_fallidos_consistencia
}

print()
print(f'Score Consistencia: {score_consistencia:.2f}%')
print(f'Registros fallidos: {registros_fallidos_consistencia:,}')

## Dimension 4: Integridad

**Pregunta:** Se mantienen las relaciones entre tablas?

Se verifica la integridad referencial con el catalogo de zonas TLC:
- `PUlocationID` debe estar en el rango 1-265 (zonas TLC validas)
- `DOlocationID` debe estar en el rango 1-265 (zonas TLC validas)

Nota: Los campos de ubicacion en FHV usan nombres en minusculas (PUlocationID, DOlocationID).

In [ ]:
print('--- Dimension 4: Integridad ---')
print()

# Rango valido de zonas TLC: 1 a 265
ZONA_MIN = 1
ZONA_MAX = 265

# Regla 1: PUlocationID debe estar en rango valido (nota: lowercase en FHV)

# --- OPTIMIZACION ONE-PASS (Auto-generada) ---
_exprs = [
    F.sum(F.when(F.col('PUlocationID').isNotNull() & ( (F.col('PUlocationID') < ZONA_MIN) | (F.col('PUlocationID') > ZONA_MAX) ), 1).otherwise(0)).alias('falla_pu'),
    F.sum(F.when(F.col('DOlocationID').isNotNull() & ( (F.col('DOlocationID') < ZONA_MIN) | (F.col('DOlocationID') > ZONA_MAX) ), 1).otherwise(0)).alias('falla_do')
]
_res = df.agg(*_exprs).collect()[0]
falla_pu = _res['falla_pu'] or 0
falla_do = _res['falla_do'] or 0
# ---------------------------------------------
# falla_pu (Optimizado en One-Pass)
print(f'  PUlocationID fuera de rango [1-265]: {falla_pu:,} registros')

# Regla 2: DOlocationID debe estar en rango valido (nota: lowercase en FHV)
# falla_do (Optimizado en One-Pass)
print(f'  DOlocationID fuera de rango [1-265]: {falla_do:,} registros')

# Registros que fallan CUALQUIERA de las reglas de integridad
df_falla_integridad = df.filter(
    (
        F.col('PUlocationID').isNotNull() &
        ((F.col('PUlocationID') < ZONA_MIN) | (F.col('PUlocationID') > ZONA_MAX))
    ) |
    (
        F.col('DOlocationID').isNotNull() &
        ((F.col('DOlocationID') < ZONA_MIN) | (F.col('DOlocationID') > ZONA_MAX))
    )
)
registros_fallidos_integridad = df_falla_integridad.count()
score_integridad = ((total_registros - registros_fallidos_integridad) / total_registros) * 100

resultados_dq['Integridad'] = {
    'score': round(score_integridad, 2),
    'descripcion': 'Porcentaje de registros con IDs de zona en el rango valido del catalogo TLC (1-265)',
    'registros_fallidos': registros_fallidos_integridad
}

print()
print(f'Score Integridad: {score_integridad:.2f}%')
print(f'Registros fallidos: {registros_fallidos_integridad:,}')

## Dimension 5: Razonabilidad

**Pregunta:** Los valores estan dentro de rangos esperados?

Para FHV solo se dispone de datos temporales. Se verifica:
- Duracion del viaje entre 1 minuto (60 segundos) y 12 horas (43,200 segundos)
- La hora de recogida debe estar en rango valido (0-23)

In [ ]:
print('--- Dimension 5: Razonabilidad ---')
print()

DURACION_MIN_SEG = 60       # 1 minuto
DURACION_MAX_SEG = 43200    # 12 horas

# Calcular duracion en segundos y hora de recogida
df_con_duracion = df.withColumn(
    'duracion_seg',
    F.unix_timestamp('dropOff_datetime') - F.unix_timestamp('pickup_datetime')
).withColumn(
    'hora_recogida',
    F.hour('pickup_datetime')
)

# Regla 1: duracion entre 1 minuto y 12 horas

# --- OPTIMIZACION ONE-PASS (Auto-generada) ---
_exprs = [
    F.sum(F.when(F.col('duracion_seg').isNotNull() & ( (F.col('duracion_seg') < DURACION_MIN_SEG) | (F.col('duracion_seg') > DURACION_MAX_SEG) ), 1).otherwise(0)).alias('falla_duracion'),
    F.sum(F.when(F.col('hora_recogida').isNotNull() & ( (F.col('hora_recogida') < 0) | (F.col('hora_recogida') > 23) ), 1).otherwise(0)).alias('falla_hora')
]
_res = df_con_duracion.agg(*_exprs).collect()[0]
falla_duracion = _res['falla_duracion'] or 0
falla_hora = _res['falla_hora'] or 0
# ---------------------------------------------
# falla_duracion (Optimizado en One-Pass)
print(f'  Duracion fuera de rango [1 min - 12 hrs]: {falla_duracion:,} registros')

# Regla 2: hora de recogida valida (0-23)
# falla_hora (Optimizado en One-Pass)
print(f'  Hora de recogida fuera de rango [0-23]: {falla_hora:,} registros')

# Registros que fallan CUALQUIERA de las reglas de razonabilidad
df_falla_razon = df_con_duracion.filter(
    (
        F.col('duracion_seg').isNotNull() &
        ((F.col('duracion_seg') < DURACION_MIN_SEG) | (F.col('duracion_seg') > DURACION_MAX_SEG))
    ) |
    (
        F.col('hora_recogida').isNotNull() &
        ((F.col('hora_recogida') < 0) | (F.col('hora_recogida') > 23))
    )
)
registros_fallidos_razon = df_falla_razon.count()
score_razonabilidad = ((total_registros - registros_fallidos_razon) / total_registros) * 100

resultados_dq['Razonabilidad'] = {
    'score': round(score_razonabilidad, 2),
    'descripcion': 'Porcentaje de registros con duracion entre 1 min y 12 hrs, y hora de recogida valida',
    'registros_fallidos': registros_fallidos_razon
}

print()
print(f'Score Razonabilidad: {score_razonabilidad:.2f}%')
print(f'Registros fallidos: {registros_fallidos_razon:,}')

## Dimension 6: Oportunidad

**Pregunta:** Los datos estan actualizados?

Se verifica que los registros correspondan al rango temporal esperado para el conjunto de datos FHV:
- Rango valido: 2019-2025 (FHV inicio reporte TLC 2015, pero solo se dispone de datos desde 2019)
- Se contabilizan registros por anio para identificar la distribucion temporal

In [ ]:
print('--- Dimension 6: Oportunidad ---')
print()

ANIO_MIN = 2019
ANIO_MAX = 2025

# Extraer anio del pickup_datetime
df_con_anio = df.withColumn('anio_recogida', F.year('pickup_datetime'))

# Distribucion por anio
print('  Distribucion de registros por anio de recogida:')
dist_anio = df_con_anio.groupBy('anio_recogida').count().orderBy('anio_recogida')
dist_anio.show(truncate=False)

# Registros fuera del rango valido
registros_fallidos_oportunidad = df_con_anio.filter(
    F.col('anio_recogida').isNull() |
    (F.col('anio_recogida') < ANIO_MIN) |
    (F.col('anio_recogida') > ANIO_MAX)
).count()

score_oportunidad = ((total_registros - registros_fallidos_oportunidad) / total_registros) * 100

resultados_dq['Oportunidad'] = {
    'score': round(score_oportunidad, 2),
    'descripcion': f'Porcentaje de registros con anio de recogida en el rango [{ANIO_MIN}-{ANIO_MAX}]',
    'registros_fallidos': registros_fallidos_oportunidad
}

print(f'Registros fuera del rango [{ANIO_MIN}-{ANIO_MAX}]: {registros_fallidos_oportunidad:,}')
print()
print(f'Score Oportunidad: {score_oportunidad:.2f}%')

## Dimension 7: Unicidad

**Pregunta:** Existen registros duplicados?

Se detectan registros duplicados agrupando por la combinacion de campos que identifican unicamente un viaje FHV:
- `pickup_datetime`
- `dropOff_datetime`
- `PUlocationID`
- `DOlocationID`
- `dispatching_base_num`

In [ ]:
print('--- Dimension 7: Unicidad ---')
print()

# Campos que definen la unicidad de un viaje FHV
campos_unicidad = [
    'pickup_datetime',
    'dropOff_datetime',
    'PUlocationID',
    'DOlocationID',
    'dispatching_base_num'
]

# Contar ocurrencias por combinacion de campos
df_agrupado = df.groupBy(campos_unicidad).count()

# Registros duplicados: grupos con mas de 1 ocurrencia
df_duplicados = df_agrupado.filter(F.col('count') > 1)
grupos_duplicados = df_duplicados.count()

# Total de registros excedentes que son duplicados
excedentes_resultado = df_duplicados.agg(
    F.sum(F.col('count') - 1).alias('excedentes')
).collect()[0]['excedentes']

registros_duplicados_total = int(excedentes_resultado) if excedentes_resultado is not None else 0

registros_unicos = total_registros - registros_duplicados_total
score_unicidad = (registros_unicos / total_registros) * 100

print(f'  Grupos de registros duplicados encontrados: {grupos_duplicados:,}')
print(f'  Total de registros excedentes (duplicados): {registros_duplicados_total:,}')
print(f'  Registros unicos: {registros_unicos:,}')

resultados_dq['Unicidad'] = {
    'score': round(score_unicidad, 2),
    'descripcion': 'Porcentaje de registros unicos (sin duplicados) por combinacion de campos clave',
    'registros_fallidos': registros_duplicados_total
}

print()
print(f'Score Unicidad: {score_unicidad:.2f}%')

## Dimension 8: Validez

**Pregunta:** Los formatos son correctos?

Se verifica que los campos clave tengan formatos validos:
- `dispatching_base_num`: longitud entre 4 y 6 caracteres
- `pickup_datetime`: anio mayor a 2000 (descarta fechas epoch o nulos disfrazados)
- `PUlocationID`: debe ser mayor a 0

In [ ]:
print('--- Dimension 8: Validez ---')
print()

# Regla 1: dispatching_base_num longitud entre 4 y 6 caracteres

# --- OPTIMIZACION ONE-PASS (Auto-generada) ---
_exprs = [
    F.sum(F.when(F.col('dispatching_base_num').isNotNull() & ( (F.length('dispatching_base_num') < 4) | (F.length('dispatching_base_num') > 6) ), 1).otherwise(0)).alias('falla_len_base'),
    F.sum(F.when(F.col('pickup_datetime').isNotNull() & (F.year('pickup_datetime') <= 2000), 1).otherwise(0)).alias('falla_anio_pickup'),
    F.sum(F.when(F.col('PUlocationID').isNotNull() & (F.col('PUlocationID') <= 0), 1).otherwise(0)).alias('falla_pu_positivo')
]
_res = df.agg(*_exprs).collect()[0]
falla_len_base = _res['falla_len_base'] or 0
falla_anio_pickup = _res['falla_anio_pickup'] or 0
falla_pu_positivo = _res['falla_pu_positivo'] or 0
# ---------------------------------------------
# falla_len_base (Optimizado en One-Pass)
print(f'  dispatching_base_num con longitud fuera de [4-6]: {falla_len_base:,} registros')

# Regla 2: anio de pickup_datetime mayor a 2000
# falla_anio_pickup (Optimizado en One-Pass)
print(f'  pickup_datetime con anio <= 2000 (posible epoch o error): {falla_anio_pickup:,} registros')

# Regla 3: PUlocationID debe ser mayor a 0 (nota: lowercase en FHV)
# falla_pu_positivo (Optimizado en One-Pass)
print(f'  PUlocationID <= 0: {falla_pu_positivo:,} registros')

# Registros que fallan CUALQUIERA de las reglas de validez
df_falla_validez = df.filter(
    (
        F.col('dispatching_base_num').isNotNull() &
        ((F.length('dispatching_base_num') < 4) | (F.length('dispatching_base_num') > 6))
    ) |
    (
        F.col('pickup_datetime').isNotNull() &
        (F.year('pickup_datetime') <= 2000)
    ) |
    (
        F.col('PUlocationID').isNotNull() &
        (F.col('PUlocationID') <= 0)
    )
)
registros_fallidos_validez = df_falla_validez.count()
score_validez = ((total_registros - registros_fallidos_validez) / total_registros) * 100

resultados_dq['Validez'] = {
    'score': round(score_validez, 2),
    'descripcion': 'Porcentaje de registros con formatos correctos en campos clave',
    'registros_fallidos': registros_fallidos_validez
}

print()
print(f'Score Validez: {score_validez:.2f}%')
print(f'Registros fallidos: {registros_fallidos_validez:,}')

## Resumen Final: Puntuacion Global de Calidad de Datos

Se presenta el resumen de las 8 dimensiones evaluadas con su puntuacion individual y la puntuacion global promedio.

**Escala de colores:**
- Verde: score >= 95% (calidad alta)
- Naranja: score >= 80% (calidad media)
- Rojo: score < 80% (calidad baja)

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType

print('=' * 60)
print('RESUMEN FINAL: CALIDAD DE DATOS FHV')
print('=' * 60)
print()

# Construir lista de resultados
filas_resumen = []
for dimension, valores in resultados_dq.items():
    filas_resumen.append((dimension, float(valores['score']), int(valores['registros_fallidos'])))
    print(f'  {dimension:<18} | Score: {valores["score"]:6.2f}% | Fallidos: {valores["registros_fallidos"]:>12,}')

print()

# Score global (promedio de las 8 dimensiones)
scores = [v['score'] for v in resultados_dq.values()]
score_global = sum(scores) / len(scores)
print(f'  Score Global (promedio): {score_global:.2f}%')
print()

# Crear DataFrame de resumen con PySpark
schema_resumen = StructType([
    StructField('Dimension', StringType(), True),
    StructField('Score', FloatType(), True),
    StructField('Registros_Fallidos', IntegerType(), True)
])
df_resumen = spark.createDataFrame(filas_resumen, schema=schema_resumen)
print('Tabla de resultados:')
df_resumen.show(truncate=False)

# --- Grafico de barras horizontales ---
dimensiones = [f[0] for f in filas_resumen]
scores_lista = [f[1] for f in filas_resumen]

colores = []
for s in scores_lista:
    if s >= 95:
        colores.append('#2ecc71')   # Verde
    elif s >= 80:
        colores.append('#f39c12')   # Naranja
    else:
        colores.append('#e74c3c')   # Rojo

fig, ax = plt.subplots(figsize=(11, 6))
y_pos = range(len(dimensiones))
barras = ax.barh(y_pos, scores_lista, color=colores, edgecolor='white', height=0.6)

# Etiquetas de valor en las barras
for barra, score in zip(barras, scores_lista):
    ax.text(
        barra.get_width() + 0.3,
        barra.get_y() + barra.get_height() / 2,
        f'{score:.1f}%',
        va='center',
        ha='left',
        fontsize=10,
        fontweight='bold'
    )

ax.set_yticks(y_pos)
ax.set_yticklabels(dimensiones, fontsize=11)
ax.set_xlabel('Score de Calidad (%)', fontsize=11)
ax.set_title(
    f'Calidad de Datos FHV - 8 Dimensiones\nScore Global: {score_global:.2f}%',
    fontsize=13,
    fontweight='bold'
)
ax.set_xlim(0, 108)
ax.axvline(x=95, color='#2ecc71', linestyle='--', alpha=0.6, linewidth=1)
ax.axvline(x=80, color='#f39c12', linestyle='--', alpha=0.6, linewidth=1)

# Leyenda
leyenda = [
    mpatches.Patch(color='#2ecc71', label='Alta (>= 95%)'),
    mpatches.Patch(color='#f39c12', label='Media (>= 80%)'),
    mpatches.Patch(color='#e74c3c', label='Baja (< 80%)')
]
ax.legend(handles=leyenda, loc='lower right', fontsize=9)

ax.invert_yaxis()
plt.tight_layout()
plt.savefig('/home/jovyan/work/data/outputs/dq_fhv_8_dimensiones.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico guardado en: /home/jovyan/work/data/outputs/dq_fhv_8_dimensiones.png')

spark.stop()
print()
print('Sesion de Spark finalizada.')